# Quantum Error Mitigation Selector (qemsel)

Welcome to the demonstration notebook for `qemsel`. This notebook showcases the high-level APIs to automatically select, optimize, and execute quantum circuits under noise-mitigated conditions.

### Core Concepts:
1. **`MitigatedExecutor`:** High-level wrapper that automatically queries the AI selector, extracts static features, maps target backend errors, and executes the mitigated circuits.
2. **Dynamic Calibration Drift:** Demonstrates how the AI selector routes around heavy mitigation (like ZNE/CDR) when noise rates degrade beyond the help-harm boundary.
3. **Graph Circuit DAG (Angle 4):** Illustrates the GNN graph representations of quantum circuits.

## 1. Setup and Initialization

In [ ]:
import os
from qiskit import QuantumCircuit
from qemsel.api import MitigatedExecutor

# Locate the trained model bundle
model_path = os.path.join("results", "boundary", "model_significant.joblib")
print(f"Using model path: {model_path}")

executor = MitigatedExecutor(model_path)

## 2. Execute a Mitigated Quantum Circuit

We create a 3-qubit GHZ state and run it using the executor. The selector will predict the best error mitigation technique (e.g. `raw`, `rem`, `zne`) based on the gate structure and the active noise parameters of the target backend.

In [ ]:
# Define a 3-qubit GHZ circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)

# Execute with cost-aware QEM recommendations
result = executor.execute(
    circuit=qc,
    pauli="ZZZ",
    backend_name="FakeLagosV2",
    base_shots=1024,
    seed=42
)

print("AI Recommended technique:", result["technique"])
print("Mitigated expectation value <ZZZ>:", result["value"])
print("Recommendation probabilities:", result["probabilities"])

## 3. Circuit DAG Graph Representation (Angle 4 GNN)

Here we show how `convert_circuit_to_graph` translates the quantum circuit into a Directed Acyclic Graph (DAG) for processing by a Graph Neural Network (GNN).

In [ ]:
from qemsel.features import convert_circuit_to_graph

graph = convert_circuit_to_graph(qc)

print("Graph Nodes (Operation Gates):")
for node in graph["nodes"]:
    print(f"  Node {node['id']}: op={node['op']}, qubits={node['qargs']}")

print("\nGraph Edges (Directed Data Flow DAG):")
print("  Edge indices:", graph["edge_index"])